# C-MAPSS Multi-seed 3-way 모델 비교

## 목적
BiLSTM / DLinear / iTransformer 세 모델의 C-MAPSS FD001~FD004 성능을 multi-seed 기반으로 시각화하고,
**Sensor correlation 분석 예측이 실제 결과와 일치하는지** 검증한다.

## 시각화 항목
1. **3-way RMSE / NASA Score / MAE bar chart** (모델 × FD, error bar = std)
2. **Multi-seed variance box plot** (모델별 seed 분포)
3. **Sensor correlation 예측 vs 실제 격차** (산점도)
4. **Parameter count vs Performance trade-off**
5. **Model selection heatmap** (어떤 FD엔 어떤 모델이 좋은가)

## 입력
MLflow `hybridpdm` experiment 의 모든 regression run.
`scripts/training/analyze_results.py` 와 동일한 fetch 로직 사용.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')

from scripts.training.analyze_results import fetch_runs, summarize, compute_gaps, extract_insights

## 1. 데이터 로드

In [ ]:
df = fetch_runs()
print(f'Total regression runs: {len(df)}')
print(f'Models: {sorted(df["model"].unique())}')
print(f'Subsets: {sorted(df["subset"].unique())}')
print(f'Seeds: {sorted(df["seed"].unique())}')
print()
df.head(20)

## 2. 3-way RMSE Bar Chart (mean +/- std)

모델 × FD 매트릭스. error bar 는 multi-seed std.

In [ ]:
def plot_metric_bar(df, metric, title=None, ax=None, lower_better=True):
    if ax is None:
        fig, ax = plt.subplots(figsize=(11, 5))
    subsets = sorted(df['subset'].unique())
    models = [m for m in ['lstm', 'dlinear', 'itransformer'] if m in df['model'].unique()]
    x = np.arange(len(subsets))
    width = 0.25
    colors = {'lstm': '#1f77b4', 'dlinear': '#ff7f0e', 'itransformer': '#2ca02c'}
    
    for i, model in enumerate(models):
        means, stds = [], []
        for sub in subsets:
            data = df[(df['model'] == model) & (df['subset'] == sub)][metric].dropna()
            means.append(data.mean() if len(data) > 0 else 0)
            stds.append(data.std() if len(data) > 1 else 0)
        offset = (i - (len(models) - 1) / 2) * width
        ax.bar(x + offset, means, width, yerr=stds, label=model,
               color=colors.get(model, 'gray'), capsize=4, edgecolor='black', linewidth=0.5)
    
    ax.set_xticks(x)
    ax.set_xticklabels(subsets)
    ax.set_xlabel('Subset')
    ax.set_ylabel(metric)
    ax.set_title(title or f'{metric} (mean +/- std) - {("lower" if lower_better else "higher")} is better')
    ax.legend(title='Model')
    return ax

fig, axes = plt.subplots(1, 2, figsize=(20, 6))
plot_metric_bar(df, 'rmse', 'RMSE (cycles) — lower is better', ax=axes[0])
plot_metric_bar(df, 'mae', 'MAE (cycles) — lower is better', ax=axes[1])
plt.tight_layout()
plt.show()

## 3. NASA Score (의사결정 비용 metric)

NASA PHM Score 는 RMSE 와 달리 late prediction 에 강한 패널티 (PdM 실무 표준).
RMSE 격차보다 NASA Score 격차가 보통 더 크게 나타남.

In [ ]:
if 'nasa_score_sum' in df.columns and df['nasa_score_sum'].notna().any():
    plot_metric_bar(df, 'nasa_score_sum', 'NASA Score (sum) - decision cost metric, lower is better')
    plt.show()
else:
    print('nasa_score_sum 데이터 없음 - reeval 후 다시 실행 필요')

## 4. Multi-seed Variance Box Plot

각 모델의 seed 별 RMSE 분포. 박스가 짧을수록 안정적.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=df, x='subset', y='rmse', hue='model', ax=ax,
            palette={'lstm': '#1f77b4', 'dlinear': '#ff7f0e', 'itransformer': '#2ca02c'})
sns.stripplot(data=df, x='subset', y='rmse', hue='model', dodge=True,
              palette={'lstm': '#1f77b4', 'dlinear': '#ff7f0e', 'itransformer': '#2ca02c'},
              size=8, edgecolor='black', linewidth=1, ax=ax, legend=False)
ax.set_title('RMSE distribution across seeds (per model x FD)')
ax.set_xlabel('Subset')
ax.set_ylabel('RMSE')
plt.tight_layout()
plt.show()

## 5. Sensor Correlation 예측 vs 실제 격차 검증

**가설**:
- mean |corr| 강한 FD (FD002=0.83, FD004=0.83) → cross-channel 모델(iTransformer) 우위
- dynamic change 강한 FD (FD003=0.148, FD001=0.115) → temporal 모델(BiLSTM) 우위
- DLinear (channel-independent) → mean |corr| 강한 FD 에서 가장 큰 손해

**검증**: mean |corr| × DLinear-BiLSTM gap (%) 산점도

In [ ]:
# Sensor correlation 분석 결과 (notebooks/sensor_correlation_analysis.ipynb 출력)
correlation_stats = pd.DataFrame({
    'subset':       ['FD001', 'FD002', 'FD003', 'FD004'],
    'mean_abs_corr': [0.5834, 0.8278, 0.4796, 0.8279],
    'dyn_change':   [0.1153, 0.0052, 0.1476, 0.0058],
    'mi_total':     [4.8249, 3.3243, 5.4974, 3.6373],
})

# 모델별 평균 RMSE
perf = df.groupby(['subset', 'model'])['rmse'].mean().unstack('model').reset_index()
perf = perf.merge(correlation_stats, on='subset')

# DLinear vs BiLSTM gap
if 'dlinear' in perf.columns and 'lstm' in perf.columns:
    perf['dlinear_vs_lstm_pct'] = (perf['dlinear'] - perf['lstm']) / perf['lstm'] * 100
if 'itransformer' in perf.columns and 'lstm' in perf.columns:
    perf['itransformer_vs_lstm_pct'] = (perf['itransformer'] - perf['lstm']) / perf['lstm'] * 100

print(perf.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (a) mean |corr| vs DLinear gap (예측: corr 강할수록 DLinear 손해 큼)
if 'dlinear_vs_lstm_pct' in perf.columns:
    ax = axes[0]
    ax.scatter(perf['mean_abs_corr'], perf['dlinear_vs_lstm_pct'], s=200, c='#ff7f0e', edgecolor='black')
    for _, row in perf.iterrows():
        ax.annotate(row['subset'], (row['mean_abs_corr'], row['dlinear_vs_lstm_pct']),
                    fontsize=11, ha='left', va='bottom', xytext=(5, 5), textcoords='offset points')
    # 회귀선
    if len(perf) >= 2:
        z = np.polyfit(perf['mean_abs_corr'], perf['dlinear_vs_lstm_pct'], 1)
        x_line = np.linspace(perf['mean_abs_corr'].min(), perf['mean_abs_corr'].max(), 50)
        ax.plot(x_line, np.poly1d(z)(x_line), 'r--', alpha=0.5, label=f'slope={z[0]:+.1f}')
        ax.legend()
    ax.set_xlabel('mean |corr| (sensor cross-correlation)')
    ax.set_ylabel('DLinear vs BiLSTM RMSE gap (%)')
    ax.set_title('Hypothesis 1: stronger sensor correlation -> larger DLinear loss')

# (b) dynamic change vs iTransformer gap (예측: dynamic 강할수록 iTransformer 손해)
if 'itransformer_vs_lstm_pct' in perf.columns:
    ax = axes[1]
    valid = perf.dropna(subset=['itransformer_vs_lstm_pct'])
    if len(valid) > 0:
        ax.scatter(valid['dyn_change'], valid['itransformer_vs_lstm_pct'], s=200, c='#2ca02c', edgecolor='black')
        for _, row in valid.iterrows():
            ax.annotate(row['subset'], (row['dyn_change'], row['itransformer_vs_lstm_pct']),
                        fontsize=11, ha='left', va='bottom', xytext=(5, 5), textcoords='offset points')
        ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
        ax.set_xlabel('dynamic change (early vs late RUL corr)')
        ax.set_ylabel('iTransformer vs BiLSTM RMSE gap (%)')
        ax.set_title('Hypothesis 2: stronger temporal dynamics -> iTransformer worse than BiLSTM')
    else:
        ax.text(0.5, 0.5, 'iTransformer data not yet available', ha='center', va='center',
                transform=ax.transAxes, fontsize=14)
        ax.set_title('Hypothesis 2 - awaiting iTransformer training')

plt.tight_layout()
plt.show()

## 6. Parameter Count vs Performance Trade-off

어떤 모델이 'cheap vs accurate' Pareto 곡선에서 어디 위치하는가.

In [ ]:
# 모델별 파라미터 수 (C-MAPSS F=14, L=30 기준)
param_counts = {
    'dlinear':      883,
    'lstm':         551234,
    'itransformer': 401026,
}

fig, ax = plt.subplots(figsize=(11, 7))
model_means = df.groupby('model')['rmse'].agg(['mean', 'std']).reset_index()

for _, row in model_means.iterrows():
    m = row['model']
    if m not in param_counts:
        continue
    ax.errorbar(
        param_counts[m], row['mean'], yerr=row['std'],
        fmt='o', markersize=15, capsize=5, label=m,
        color={'lstm': '#1f77b4', 'dlinear': '#ff7f0e', 'itransformer': '#2ca02c'}.get(m, 'gray'),
        markeredgecolor='black', markeredgewidth=1.5,
    )
    ax.annotate(
        f'{m}\n{param_counts[m]:,} params\nRMSE {row["mean"]:.2f}',
        (param_counts[m], row['mean']),
        fontsize=10, ha='left', va='bottom', xytext=(10, 10), textcoords='offset points',
    )

ax.set_xscale('log')
ax.set_xlabel('Parameter count (log scale)')
ax.set_ylabel('Average RMSE across FD001-004')
ax.set_title('Parameter count vs Performance trade-off')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Model Selection Heatmap

각 (모델, FD) 조합의 성능을 정규화 점수(0-1, 낮을수록 좋음)로 시각화.
어떤 FD 에 어떤 모델을 써야 하는가 한눈에 파악.

In [ ]:
pivot = df.groupby(['subset', 'model'])['rmse'].mean().unstack('model')
# subset 별 normalize (min-max)
pivot_norm = pivot.sub(pivot.min(axis=1), axis=0).div(
    pivot.max(axis=1) - pivot.min(axis=1), axis=0
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn_r', ax=axes[0],
            cbar_kws={'label': 'RMSE'})
axes[0].set_title('Absolute RMSE per (FD x model)')

sns.heatmap(pivot_norm, annot=True, fmt='.2f', cmap='RdYlGn_r', ax=axes[1],
            cbar_kws={'label': 'Normalized RMSE (0=best, 1=worst per FD)'})
axes[1].set_title('Normalized RMSE — best model per FD highlighted')

plt.tight_layout()
plt.show()

## 8. 종합 인사이트

In [ ]:
print('=== Auto Insights ===')
for line in extract_insights(df):
    print(f'  - {line}')

print()
print('=== Per-subset best model (RMSE) ===')
best = pivot.idxmin(axis=1).to_dict()
for sub, model in best.items():
    rmse = pivot.loc[sub, model]
    print(f'  {sub}: {model} (RMSE {rmse:.2f})')

## 9. 가설 검증 요약

Sensor correlation 분석으로 세웠던 가설들이 검증됐는가?

| 가설 | 예측 | 검증 결과 (학습 완료 후) |
|---|---|---|
| H1: mean \|corr\| 강함 → DLinear 손해 ↑ | FD002/FD004 에서 DLinear 격차 최대 | 표 참조 |
| H2: dynamic 강함 → BiLSTM 우위 (iTransformer 손해) | FD001/FD003 에서 iTransformer 격차 양수 | 표 참조 |
| H3: BiLSTM 의 FD003 절대 RMSE 최저 | dynamic 강함 + temporal attention 정합 | 검증 |
| H4: NASA Score gap > RMSE gap | DLinear late prediction 편향 | 1, 2 셀 비교 |